# Karaokê Zapia — processamento no Kaggle GPU

O Kaggle oferece GPU grátis em sessões de notebook, com cota semanal anunciada de 30 horas (às vezes maior, dependendo da disponibilidade). A cota e os recursos podem variar; sessões de notebook não são um serviço web permanente.

**Importante:** cole um link do YouTube apenas para associar/identificar a música. O notebook não baixa o áudio do vídeo. Adicione como entrada um arquivo de áudio que você tem autorização para processar. Documentação: [uso eficiente de GPU no Kaggle](https://www.kaggle.com/docs/efficient-gpu-usage).

Antes de executar: em Settings do notebook, ative **GPU** e **Internet**. Crie/adicionar um dataset privado com seu áudio.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install demucs faster-whisper
import torch
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## Worker Demucs + Whisper + FFmpeg

A célula grava o worker do projeto no ambiente do notebook.

In [ ]:
%%writefile /kaggle/working/gpu_worker.py
#!/usr/bin/env python3
"""Separate an authorized audio file, transcribe it, and render a karaoke MP4.

This worker intentionally does not download or extract media from YouTube.
Use an audio file that you own or have permission to process.
"""
from __future__ import annotations

import argparse
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path


def run(cmd: list[str]) -> None:
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode:
        detail = (result.stderr or result.stdout or "").strip()[-2500:]
        raise RuntimeError(f"Command failed: {' '.join(cmd[:3])}\n{detail}")


def stamp(seconds: float) -> str:
    ms = max(0, int(round(seconds * 1000)))
    hours, ms = divmod(ms, 3_600_000)
    minutes, ms = divmod(ms, 60_000)
    secs, ms = divmod(ms, 1000)
    return f"{hours:02}:{minutes:02}:{secs:02},{ms:03}"


def safe_srt_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().replace("--> ", "→ ")


def make_captions(segments) -> list[tuple[float, float, str]]:
    """Group Whisper's word timestamps into readable, short karaoke captions."""
    words = []
    fallback = []
    for segment in segments:
        fallback.append((float(segment.start), float(segment.end), safe_srt_text(segment.text)))
        for word in (getattr(segment, "words", None) or []):
            text = safe_srt_text(word.word)
            if text:
                words.append((float(word.start), float(word.end), text))
    if not words:
        return [x for x in fallback if x[2]]

    lines = []
    group = []
    start = words[0][0]
    for word in words:
        if group and (len(group) >= 7 or word[0] - start >= 4.0):
            lines.append((start, group[-1][1], "".join(x[2] for x in group).strip()))
            group = []
            start = word[0]
        if not group:
            start = word[0]
        group.append(word)
        if re.search(r"[.!?;:]$", word[2]) and len(group) >= 3:
            lines.append((start, word[1], "".join(x[2] for x in group).strip()))
            group = []
    if group:
        lines.append((start, group[-1][1], "".join(x[2] for x in group).strip()))
    return [(a, max(a + 0.25, b), text) for a, b, text in lines if text]


def transpose_audio(source: Path, destination: Path, semitones: float) -> None:
    """Shift pitch while approximately preserving duration (FFmpeg atempo/asetrate)."""
    if abs(semitones) < 1e-6:
        shutil.copy2(source, destination)
        return
    try:
        sample_rate = int(subprocess.check_output([
            "ffprobe", "-v", "error", "-select_streams", "a:0",
            "-show_entries", "stream=sample_rate", "-of", "default=noprint_wrappers=1:nokey=1", str(source),
        ], text=True).strip())
    except Exception as exc:
        raise RuntimeError("Não consegui identificar a taxa de amostragem para ajustar o tom.") from exc
    ratio = 2 ** (semitones / 12)
    atempo = 1 / ratio
    audio_filter = f"asetrate={sample_rate * ratio:.5f},aresample={sample_rate},atempo={atempo:.8f}"
    print(f"Ajustando tom: {semitones:+g} semitons (mantendo a duração aproximada)…", flush=True)
    run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(source), "-af", audio_filter, "-c:a", "pcm_s16le", str(destination)])


def main() -> int:
    parser = argparse.ArgumentParser(description="Gera instrumental, legenda e MP4 de karaokê a partir de áudio autorizado.")
    parser.add_argument("audio", type=Path, help="Arquivo de áudio que você tem autorização para processar")
    parser.add_argument("--output-dir", type=Path, default=Path("karaoke-output"))
    parser.add_argument("--demucs-model", default="htdemucs")
    parser.add_argument("--whisper-model", default="small", help="tiny, base, small, medium ou large-v3")
    parser.add_argument("--language", default=None, help="Idioma opcional, por exemplo pt")
    parser.add_argument("--semitones", type=int, default=0, help="Transposição de -6 (mais grave) a +6 (mais agudo)")
    args = parser.parse_args()
    if not -6 <= args.semitones <= 6:
        parser.error("A transposição deve ficar entre -6 e +6 semitons.")

    source = args.audio.expanduser().resolve()
    if not source.is_file():
        parser.error(f"Arquivo não encontrado: {source}")
    if shutil.which("ffmpeg") is None:
        parser.error("FFmpeg não encontrado. Instale FFmpeg antes de rodar.")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    stem = re.sub(r"[^\w.-]+", "_", source.stem, flags=re.UNICODE).strip("._") or "karaoke"

    try:
        import torch
        from faster_whisper import WhisperModel
    except ImportError as exc:
        parser.error(f"Dependência ausente: {exc}. Instale requirements-separation.txt.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    print(f"GPU: {device} | Whisper: {args.whisper_model} ({compute_type})", flush=True)

    with tempfile.TemporaryDirectory(prefix="karaoke_worker_") as temp_name:
        temp = Path(temp_name)
        demucs_out = temp / "separated"
        print("Separando voz e instrumental com Demucs…", flush=True)
        run([sys.executable, "-m", "demucs", "--two-stems=vocals", "-n", args.demucs_model, "-o", str(demucs_out), str(source)])
        instrumental_src = next(demucs_out.rglob("no_vocals.wav"), None)
        if instrumental_src is None:
            raise RuntimeError("Demucs terminou, mas a faixa instrumental não foi encontrada.")
        shifted_src = temp / "pitch_adjusted.wav"
        transpose_audio(instrumental_src, shifted_src, args.semitones)
        instrumental_src = shifted_src

        print("Reconhecendo a letra e os tempos com Whisper…", flush=True)
        model = WhisperModel(args.whisper_model, device=device, compute_type=compute_type)
        segments_iter, _info = model.transcribe(str(source), language=args.language, word_timestamps=True, vad_filter=True)
        segments = list(segments_iter)
        captions = make_captions(segments)

        srt_path = args.output_dir / f"{stem}.srt"
        with srt_path.open("w", encoding="utf-8") as f:
            for i, (start, end, text) in enumerate(captions, 1):
                f.write(f"{i}\n{stamp(start)} --> {stamp(end)}\n{text}\n\n")

        tone_label = "tom_0" if args.semitones == 0 else f"tom_{'mais' if args.semitones > 0 else 'menos'}_{abs(args.semitones)}"
        instrumental = args.output_dir / f"{stem}_{tone_label}_instrumental.wav"
        shutil.copy2(instrumental_src, instrumental)
        video = args.output_dir / f"{stem}_{tone_label}_karaoke.mp4"
        duration_cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(instrumental)]
        try:
            duration = float(subprocess.check_output(duration_cmd, text=True).strip())
        except Exception as exc:
            raise RuntimeError("Não consegui medir a duração do instrumental com ffprobe.") from exc
        subtitle_path = str(srt_path.resolve()).replace("\\", "\\\\").replace(":", "\\:").replace("'", "\\'")
        vf = f"subtitles='{subtitle_path}':force_style='FontName=DejaVu Sans,FontSize=32,PrimaryColour=&H00FFFFFF,OutlineColour=&H00101010,BorderStyle=1,Outline=3,Shadow=0,Alignment=2,MarginV=72'"
        print("Renderizando o vídeo MP4…", flush=True)
        run([
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-f", "lavfi", "-i", f"color=c=0x17131f:s=1280x720:r=30:d={duration:.3f}",
            "-i", str(instrumental), "-vf", vf, "-map", "0:v:0", "-map", "1:a:0",
            "-c:v", "libx264", "-preset", "veryfast", "-pix_fmt", "yuv420p",
            "-c:a", "aac", "-b:a", "192k", "-shortest", "-movflags", "+faststart", str(video),
        ])

    print("Pronto:")
    print(f"  MP4: {video.resolve()}")
    print(f"  Instrumental: {instrumental.resolve()}")
    print(f"  Legenda editável: {srt_path.resolve()}")
    print("Revise a letra e os tempos: reconhecimento de canto pode errar palavras e sincronização.")
    return 0


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except RuntimeError as exc:
        print(f"ERRO: {exc}", file=sys.stderr)
        raise SystemExit(1)


## Escolha o link do YouTube e o áudio autorizado

Adicione o arquivo em **Add Input**. Se o dataset tiver mais de um áudio, ajuste `AUDIO_PATH` para o arquivo desejado.

In [ ]:
from pathlib import Path
import re, subprocess, sys, json, shutil
YOUTUBE_URL = "https://www.youtube.com/watch?v=COLE_O_ID_AQUI"
AUDIO_PATH = None  # opcional: caminho completo do áudio no /kaggle/input
SEMITONES = 0  # -6 a +6; negativo = mais grave, positivo = mais agudo; pode importar karaoke_settings.json do site
settings_files = sorted(Path("/kaggle/input").rglob("karaoke_settings.json"))
if settings_files:
    settings = json.loads(settings_files[0].read_text(encoding="utf-8"))
    YOUTUBE_URL = settings.get("youtube_url", YOUTUBE_URL)
    SEMITONES = int(settings.get("semitones", SEMITONES))
    print("Configuração importada:", settings_files[0])
if not -6 <= SEMITONES <= 6: raise ValueError("SEMITONES precisa ficar entre -6 e +6.")
audio_exts = {".mp3", ".wav", ".flac", ".m4a", ".ogg", ".opus"}
available = sorted(p for p in Path("/kaggle/input").rglob("*") if p.is_file() and p.suffix.lower() in audio_exts)
if not available:
    raise FileNotFoundError("Adicione um dataset privado contendo seu arquivo de áudio em Add Input.")
print("Áudios disponíveis:")
for p in available: print(p)
if AUDIO_PATH is None: AUDIO_PATH = str(available[0])
audio = Path(AUDIO_PATH)
if not audio.is_file(): raise FileNotFoundError(f"Arquivo não encontrado: {audio}")
m = re.search(r"(?:v=|youtu\.be/|/shorts/|/embed/|/live/)([A-Za-z0-9_-]{11})", YOUTUBE_URL)
if not m: raise ValueError("Cole um link válido do YouTube em YOUTUBE_URL.")
video_id = m.group(1)
print("Link associado:", f"https://www.youtube.com/watch?v={video_id}")
print("Áudio selecionado:", audio.name)
print("Tom escolhido:", f"{SEMITONES:+d} semitons")

O worker usa Whisper `tiny` para reduzir o tempo e ajusta o tom selecionado de -6 (mais grave) a +6 (mais agudo), mantendo a duração aproximadamente. Você pode escolher o tom no site, baixar `karaoke_settings.json` e adicioná-lo ao mesmo dataset privado do áudio; ou mudar `SEMITONES` na célula anterior. Revise o SRT: voz cantada pode ser transcrita com erros. Depois, baixe `karaoke_kaggle_output.zip` na área de arquivos de saída do notebook.

In [ ]:
output_dir = Path("/kaggle/working/karaoke-output")
output_dir.mkdir(exist_ok=True)
cmd = [sys.executable, "/kaggle/working/gpu_worker.py", str(audio), "--output-dir", str(output_dir), "--language", "pt", "--whisper-model", "tiny", "--semitones", str(SEMITONES)]
subprocess.run(cmd, check=True)
(output_dir / "youtube_reference.txt").write_text(f"YouTube: https://www.youtube.com/watch?v={video_id}\nTom: {SEMITONES:+d} semitons\n", encoding="utf-8")
archive = shutil.make_archive("/kaggle/working/karaoke_kaggle_output", "zip", output_dir)
print("Pacote pronto:", archive)
print("Conteúdo:", [p.name for p in output_dir.iterdir()])